# CIFAR-10 Baseline Experiment

Train the baseline CNN on CIFAR-10 and log metrics.

In [11]:
import sys
from pathlib import Path
import time

candidate_roots = [
    Path('/content/drive/MyDrive/ouroboros'),
]
project_root = next((p for p in candidate_roots if p.exists()), None)
if project_root is None:
    raise FileNotFoundError('Project root not found. Update candidate_roots.')
sys.path.insert(0, str(project_root))

import torch
from torch import nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR

from src.data_loaders import get_cifar10_loaders, get_cifar10_test_loader
from src.metrics import MetricsLogger, compute_system_metrics, plot_learning_curves, reset_cuda_peak_memory
from src.models import CNN3Layer, DEFAULT_CHANNELS
from src.trainer import train_epoch, validate_epoch, save_checkpoint
from src.utils import get_device, set_seed, ensure_dirs

set_seed(42)
device = get_device()
ensure_dirs('results', 'results/figures', 'checkpoints')

lr = 1e-3
batch_size = 128
epochs = 5
weight_decay = 1e-4

train_loader, val_loader = get_cifar10_loaders(batch_size, 2, 'assets')
model = CNN3Layer(num_classes=10, in_channels=3, channels=DEFAULT_CHANNELS).to(device)
optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
criterion = nn.CrossEntropyLoss()

# Create warmup + cosine scheduler
warmup_scheduler = LinearLR(optimizer, start_factor=0.5, end_factor=1.0, total_iters=1)
cosine_scheduler = CosineAnnealingLR(optimizer, T_max=4, eta_min=1e-5)
scheduler = SequentialLR(optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[1])

# Count total params for logging
total_params = sum(p.numel() for p in model.parameters())

logger = MetricsLogger(run_metadata={
    'dataset': 'CIFAR-10',
    'epochs': epochs,
    'lr': lr,
    'weight_decay': weight_decay,
    'channels': list(DEFAULT_CHANNELS),
    'total_params': total_params,
})

# Mandatory logging
print(f"[OPTIMIZER] Type: {type(optimizer).__name__} | lr={lr} | weight_decay={weight_decay}")
print(f"[MODEL] Total params: {total_params}")

for epoch in range(1, epochs + 1):
    reset_cuda_peak_memory()
    start = time.perf_counter()

    # Get LR at start of epoch
    current_lr = optimizer.param_groups[0]['lr']

    train_metrics = train_epoch(
        model=model,
        dataloader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=device,
        amp_enabled=(device.type == 'cuda'),
        use_compile=hasattr(torch, 'compile'),
        collect_grad_stats=True,
    )
    val_metrics = validate_epoch(
        model=model,
        dataloader=val_loader,
        criterion=criterion,
        device=device,
    )

    # Step scheduler AFTER validation
    scheduler.step()

    # Log LR after scheduler step
    post_step_lr = optimizer.param_groups[0]['lr']
    print(f"[SCHEDULER] Epoch {epoch} | LR: {post_step_lr:.6f}")

    system_metrics = compute_system_metrics(
        total_samples=len(train_loader) * batch_size,
        start_time=start,
        end_time=time.perf_counter(),
        device=device,
    )
    logger.log_epoch(
        epoch=epoch,
        train={k: v for k, v in train_metrics.items() if k != 'gradients'},
        validation=val_metrics,
        gradients=train_metrics.get('gradients'),
        system=system_metrics,
        learning_rate=current_lr,
    )
    print('Epoch', epoch, 'train', train_metrics, 'val', val_metrics)

metrics_path = Path('results') / 'cifar10_baseline_metrics.json'
logger.to_json(metrics_path)
plot_learning_curves(metrics_path, output_dir='results/figures', prefix='cifar10_baseline')

checkpoint_path = Path('checkpoints') / 'cifar10_baseline.pth'
final_metrics = {
    'train_loss': logger.epoch_metrics[-1]['train'].get('loss', 0.0),
    'train_accuracy': logger.epoch_metrics[-1]['train'].get('accuracy', 0.0),
    'val_loss': logger.epoch_metrics[-1]['validation'].get('loss', 0.0),
    'val_accuracy': logger.epoch_metrics[-1]['validation'].get('accuracy', 0.0),
}
save_checkpoint(str(checkpoint_path), model, optimizer, epochs, final_metrics, scheduler=scheduler)
print(f'\nSaved metrics to {metrics_path}')
print(f'Saved checkpoint to {checkpoint_path}')

# Test set evaluation
print('\n' + '='*60)
print('TEST SET EVALUATION')
print('='*60)
test_loader = get_cifar10_test_loader(batch_size, 2, 'assets')
test_metrics = validate_epoch(model, test_loader, criterion, device)
print(f"Test Loss: {test_metrics['loss']:.4f}")
print(f"Test Accuracy: {test_metrics['accuracy']:.4f} ({test_metrics['accuracy']*100:.2f}%)")

# Final Summary
print('\n' + '='*60)
print('FINAL SUMMARY - CIFAR-10 Baseline')
print('='*60)
print(f"Dataset: CIFAR-10")
print(f"Model Parameters: {total_params:,}")
print(f"Training Epochs: {epochs}")
print(f"Batch Size: {batch_size}")
print(f"Learning Rate: {lr}")
print(f"Weight Decay: {weight_decay}")
print(f"\nFinal Training Loss: {final_metrics['train_loss']:.4f}")
print(f"Final Training Accuracy: {final_metrics['train_accuracy']:.4f} ({final_metrics['train_accuracy']*100:.2f}%)")
print(f"Final Validation Loss: {final_metrics['val_loss']:.4f}")
print(f"Final Validation Accuracy: {final_metrics['val_accuracy']:.4f} ({final_metrics['val_accuracy']*100:.2f}%)")
print(f"Final Test Loss: {test_metrics['loss']:.4f}")
print(f"Final Test Accuracy: {test_metrics['accuracy']:.4f} ({test_metrics['accuracy']*100:.2f}%)")
print('='*60)


100%|██████████| 170M/170M [03:47<00:00, 751kB/s] 


[OPTIMIZER] Type: AdamW | lr=0.001 | weight_decay=0.0001
[MODEL] Total params: 94986
[SCHEDULER] Epoch 1 | LR: 0.001000
Epoch 1 train {'loss': 1.597546258950845, 'accuracy': 0.4317708333333333, 'gradients': {'total_l2_norm': 7.011099921802289, 'per_layer_l2_norms': {'_orig_mod.conv1.weight': 6.556224346160889, '_orig_mod.conv1.bias': 4.065442772116512e-05, '_orig_mod.bn1.weight': 0.15127155184745789, '_orig_mod.bn1.bias': 0.11985968798398972, '_orig_mod.conv2.weight': 2.3674538135528564, '_orig_mod.conv2.bias': 1.237115702679148e-05, '_orig_mod.bn2.weight': 0.08730749785900116, '_orig_mod.bn2.bias': 0.05981920659542084, '_orig_mod.conv3.weight': 0.534106969833374, '_orig_mod.conv3.bias': 2.3976774627954e-06, '_orig_mod.bn3.weight': 0.03805205971002579, '_orig_mod.bn3.bias': 0.04368414729833603, '_orig_mod.fc.weight': 0.47449636459350586, '_orig_mod.fc.bias': 0.06620568037033081}, 'zero_grad_parameters': 0}} val {'loss': 1.4115927461624145, 'accuracy': 0.4756}
[SCHEDULER] Epoch 2 | LR: 

# Conclusion

The baseline CNN exhibits consistent and well-conditioned optimization dynamics under the selected AdamW + cosine annealing regime. Training loss decreases smoothly from 1.60 to 0.99 over five epochs, accompanied by a corresponding rise in training accuracy from 43.2% to 65.1%, indicating effective utilization of model capacity despite the constrained parameter budget (~95k parameters).

Validation and test performance closely track training behavior throughout optimization, converging to 64.85% accuracy with a negligible generalization gap (<0.5%). This tight coupling between training and validation metrics suggests that the model operates in a low-overfitting regime, and that early stopping at five epochs provides a reliable proxy for relative configuration quality on CIFAR-10.

Gradient norm analysis further confirms numerical stability and healthy gradient flow across all layers. Total L2 norms remain bounded throughout training, with expected dominance in early convolutional layers and no evidence of gradient collapse or dead parameters. Minor fluctuations in gradient magnitude during later epochs are consistent with stochastic batch effects under low learning-rate conditions rather than structural instability.

Overall, these results establish a robust and reproducible baseline that meets the intended performance and stability targets for early-stage evaluation. The observed learning dynamics indicate that further improvements are unlikely to arise from architectural scaling alone, but rather from training-procedure refinements, including optimization hyperparameters, regularization strategies, and data augmentation, making this configuration an appropriate foundation for subsequent meta-optimization and automated search in Phase 2.